# Intelligent Calculator

An LLM agent (Groq `openai/gpt-oss-120b` through LangChain) that chooses between two exact calculator tools, then the same agent is exposed to ChatGPT as an MCP server through an OpenAI Secure MCP Tunnel.

**How to use this notebook:** run the cells in order (Kernel -> Restart & Run All). Cell 1 needs your Groq API key. Cells 7-10 are the saved runs: save the notebook after running so their outputs are stored in the file.

| Cell | Purpose |
|---|---|
| 1 | Load the Groq API key |
| 2 | Imports and the model |
| 3 | `add` tool |
| 4 | `tree_distance` tool |
| 5 | Agent instructions and agent |
| 6 | `show_run` helper |
| 7 | Run: apples (5 + 10) |
| 8 | Run: tree distance |
| 9 | Run: missing angle |
| 10 | Run: fresh 7 + 8 |
| 11 | MCP server definition |
| 12 | Start the server and connect the tunnel |

In [1]:
%pip install -q 'langchain>=1,<2' 'langchain-groq<2'

## Cell 1: API key
The key is read from the `GROQ_API_KEY` environment variable or a local `.env` file. If neither exists, you are prompted. The key is never written into the notebook, so it is safe to commit.

In [2]:
import os
from getpass import getpass

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# Prompt for key if not found, or if you want to update it
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Groq API key: ").strip()

print("Groq key loaded:", bool(os.environ.get("GROQ_API_KEY")))

Groq API key: ··········
Groq key loaded: True


## Cell 2: Imports and model
`temperature=0` keeps tool choice repeatable, `timeout=30` stops a stuck call, and `max_retries=1` avoids long retry loops.

In [3]:
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent
import os

# Initialize model using the loaded API key
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    timeout=30,
    max_retries=1,
    groq_api_key=os.environ.get("GROQ_API_KEY")
)

## Cell 3: The `add` tool
Addition is done by code, not by the model, so the answer is exact. The docstring is what the model reads to decide when to use the tool.

In [4]:
@tool
def add(a: int, b: int) -> int:
    """Add two whole-number quantities when they are combined."""
    return a + b

print(add.invoke({"a": 5, "b": 10}))  # expected: 15

15


## Cell 4: The `tree_distance` tool
Horizontal distance on level ground is `height / tan(angle)`, where height is measured above eye level. Invalid input returns a message instead of raising an error, so the agent can ask the user for a correction.

In [5]:
from math import isfinite, radians, tan

@tool
def tree_distance(height_m: float, angle_deg: float) -> float | str:
    """Find horizontal tree distance on level ground from height above eye level in metres and angle of elevation in degrees."""
    if not isfinite(height_m) or height_m <= 0:
        return "Invalid height. Supply a positive finite height above eye level."
    if not isfinite(angle_deg) or not 0 < angle_deg < 90:
        return "Invalid angle. Supply an angle strictly between 0 and 90 degrees."
    return round(height_m / tan(radians(angle_deg)), 2)

print(tree_distance.invoke({"height_m": 12, "angle_deg": 30}))  # expected: 20.78

20.78


## Cell 5: Instructions and agent
The system prompt sets the rules: use each tool only for its job, call the matching tool once, ask for missing values instead of guessing, ask for a correction on invalid input, and give the final answer with units.

In [6]:
INSTRUCTIONS = (
    "You are the Intelligent Calculator. Use add for combining two whole-number "
    "quantities. Use tree_distance only for horizontal distance on level ground "
    "when height above eye level in metres and angle of elevation in degrees "
    "are supplied. For supported complete problems, call the matching tool once "
    "and use its result. If a required value is missing, ask for it before "
    "calling a tool. Never guess inputs. If a tool reports an invalid input, ask "
    "for a correction. For other requests, explain what you can calculate. "
    "Give the final answer with units."
)
# Recreate agent with the updated model config
agent = create_agent(model=model, tools=[add, tree_distance], system_prompt=INSTRUCTIONS)

## Cell 6: `show_run` helper
Runs the agent on one problem and prints the whole trace in order: the question, each tool call with its arguments, each tool result, and the final answer. This makes it visible *which* tool was chosen and *what* it returned. `recursion_limit=8` caps the loop.

In [7]:
def show_run(problem):
    print("QUESTION:", problem, "\n")
    result = agent.invoke(
        {"messages": [("human", problem)]},
        config={"recursion_limit": 8, "run_name": "intro_calculator"},
    )
    for m in result["messages"][1:]:
        if m.type == "ai" and getattr(m, "tool_calls", None):
            for call in m.tool_calls:
                print("TOOL CALL  :", call["name"], call["args"])
        elif m.type == "tool":
            print("TOOL RESULT:", m.content)
        elif m.type == "ai":
            print("\nFINAL ANSWER:", m.content)
    return result

## Cell 7: Saved run, apples
**Expected:** the agent calls `add` with `a=5, b=10`, the tool returns `15`, and the answer is 15 apples. Addition goes through the tool, not through the model's own arithmetic.

In [8]:
apples_run = show_run("One person brings five apples and another brings ten. How many apples do they have together?")

QUESTION: One person brings five apples and another brings ten. How many apples do they have together? 

TOOL CALL  : add {'a': 5, 'b': 10}
TOOL RESULT: 15

FINAL ANSWER: They have **15 apples** in total.


## Cell 8: Saved run, tree distance
**Expected:** the agent calls `tree_distance` with `height_m=12, angle_deg=30`, the tool returns `20.78`, and the answer is about 20.78 metres.

In [9]:
tree_run = show_run("The top of a tree is 12 metres above my eye level and the angle of elevation is 30 degrees. How far am I from the tree?")

QUESTION: The top of a tree is 12 metres above my eye level and the angle of elevation is 30 degrees. How far am I from the tree? 

TOOL CALL  : tree_distance {'angle_deg': 30, 'height_m': 12}
TOOL RESULT: 20.78

FINAL ANSWER: The horizontal distance from you to the tree is **20.78 m**.


### Explanations and Observations
- **Apples Run**: The agent correctly identified the problem as an addition query, invoked the `add` tool with parameters `a=5` and `b=10`, and returned the exact calculated total of `15 apples`. By utilizing the tool, arithmetic mistakes are avoided.
- **Tree Distance Run**: The agent parsed the height (`12m`) and angle (`30 degrees`) inputs from the prompt, correctly selected and executed the `tree_distance` tool, and returned the horizontal distance of `20.78 m`.
- **Missing Angle Run**: In accordance with the "never guess inputs" instruction, the agent did not execute any tool when the angle parameter was omitted, instead politely requesting the missing angle.

### LangSmith and MCP Integration Verification
- **LangSmith Tracing Check**: **[UNAVAILABLE]** (Not configured in this environment)
- **MCP Server Deployment Check**: **[UNAVAILABLE]** (Tunnel configuration and external client verification are pending local deployment)

In [10]:
---

QUESTION: One person brings seven apples and another brings eight. How many apples do they have together? 

TOOL CALL  : add {'a': 7, 'b': 8}
TOOL RESULT: 15

FINAL ANSWER: The total number of apples is **15 apples**.


---

## Cell 9: Saved run, missing angle
**Expected:** no tool call at all. The height is given but the angle is not, so the agent should ask for the angle of elevation instead of guessing one. This checks the "never guess inputs" rule.

In [11]:
missing_angle_run = show_run("How far away is a tree that is 15 metres above my eye level?")

QUESTION: How far away is a tree that is 15 metres above my eye level? 


FINAL ANSWER: To determine the horizontal distance to the tree, I need both:

* the height above your eye level (you’ve given 15 metres), **and**
* the angle of elevation from your eye to the top of the tree (in degrees).

Could you tell me the angle of elevation?


## Cell 10: Fresh run, 7 + 8
A new problem the earlier cells never used, so the result cannot come from a stale or remembered answer. **Expected:** `add` with `a=7, b=8`, result `15`, answer 15 apples.

In [12]:
fresh_run = show_run("One person brings seven apples and another brings eight. How many apples do they have together?")

QUESTION: One person brings seven apples and another brings eight. How many apples do they have together? 

TOOL CALL  : add {'a': 7, 'b': 8}
TOOL RESULT: 15

FINAL ANSWER: The total number of apples is **15 apples**.


## Cell 11: Expose the agent as an MCP server
`FastMCP` turns one Python function into an MCP tool. ChatGPT sees a single tool, `solve_math_problem`, and the agent above decides internally which calculator to use. `stateless_http=True` and `json_response=True` keep each request independent, `readOnlyHint=True` tells ChatGPT the tool changes nothing, and the length check rejects empty or oversized input.

This needs `mcp<2` (version 2 renamed `FastMCP`).

In [18]:
%pip install mcp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [mcp]


In [21]:
import sys
import os
import subprocess

def clean_and_install_mcp():
    print("Enforcing mcp<2 version and clearing module caches...")
    # Downgrade packages cleanly
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "mcp", "mcp-types"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mcp<2"])

    # Purge any imported mcp module cache from memory
    for key in list(sys.modules.keys()):
        if key == "mcp" or key.startswith("mcp."):
            del sys.modules[key]

try:
    from mcp.server.fastmcp import FastMCP
    from mcp.types import ToolAnnotations
except (ImportError, ModuleNotFoundError):
    clean_and_install_mcp()
    from mcp.server.fastmcp import FastMCP
    from mcp.types import ToolAnnotations

mcp = FastMCP(
    "Intelligent Calculator",
    host=os.environ.get("INTRO_MCP_HOST", "127.0.0.1"),
    stateless_http=True,
    json_response=True,
)

@mcp.tool(annotations=ToolAnnotations(readOnlyHint=True, openWorldHint=False))
def solve_math_problem(problem: str) -> str:
    """Solve an apples addition or tree-distance word problem. Pass the original wording."""
    problem = problem.strip()
    if not 1 <= len(problem) <= 1500:
        return "Enter an English word problem of 1 to 1500 characters."
    result = agent.invoke(
        {"messages": [("human", problem)]},
        config={"recursion_limit": 8, "run_name": "intro_calculator_mcp"},
    )
    return result["messages"][-1].content

print("MCP tool registered: solve_math_problem")

Enforcing mcp<2 version and clearing module caches...
MCP tool registered: solve_math_problem


## Cell 12: Serve it and connect the tunnel
Jupyter cannot keep a server running in a cell without blocking, so the server runs from the script in a terminal. Cell 12 only shows the commands.

1. **Terminal 1:** `python server.py` (serves `http://127.0.0.1:8000/mcp`).
2. **Terminal 2:** set `CONTROL_PLANE_API_KEY`, then run `tunnel-client init ...`, `tunnel-client doctor --profile calculator --explain`, and `tunnel-client run --profile calculator`. The exact commands are in `README.md`.
3. **ChatGPT:** Plugins, plus, Add custom MCP server, Connection **Tunnel**, Authentication **No authentication**.
4. Test with `@Intelligent Calculator` followed by the apples, tree, missing-angle and 7 + 8 prompts above.

In [22]:
print("python server.py            # terminal 1, listens on http://127.0.0.1:8000/mcp")
print("tunnel-client run --profile calculator   # terminal 2, keep it running")

python server.py            # terminal 1, listens on http://127.0.0.1:8000/mcp
tunnel-client run --profile calculator   # terminal 2, keep it running
